<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 1 · Estructura antes que fórmulas
**Excel Corporativo · 12 Claves** — Datos tabulares y Tablas de Excel · ⏱ 45 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/01_estructura_y_tablas.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-01)

**Al terminar podrás:**
1. Detectar con código los defectos de un reporte «para imprimir».
2. Reconstruirlo como tabla ordenada: una fila por registro.
3. Medir cuánto se equivoca una suma hecha sobre un reporte.
4. Hacer un diagnóstico de calidad a una base antes de analizarla.

**Cómo usar este cuaderno.** Lee cada explicación y ejecuta la celda de código que sigue con **Shift + Enter**. Las celdas con un título y el código oculto (*Comprobar*, *Ver una solución*) se ejecutan igual; haz doble clic sobre ellas si quieres ver su código. Si algo falla, usa *Entorno de ejecución › Reiniciar y ejecutar todo*.

## Excel y Python, lado a lado
Lo que haces en Excel tiene un equivalente en pandas, la librería de Python para datos tabulares. Este cuaderno usa esta correspondencia:

| En Excel | En pandas | Para qué |
|---|---|---|
| Una Tabla (`Ctrl+T`) | un `DataFrame` | Datos con nombre de columna y tipo |
| Separar celdas + rellenar hacia abajo | `.ffill()` | Repetir la ciudad en cada fila |
| Filtro y eliminar filas | `df[condición]` | Quitar subtotales y filas en blanco |
| Anular dinamización de columnas (Power Query) | `.melt()` | Pasar T1…T4 a una columna *trimestre* |
| `SUMA`, `SUMAR.SI.CONJUNTO` | `.sum()`, `.groupby().sum()` | Totales |

In [ ]:
# @title Preparar el entorno (ejecuta esta celda primero)
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"   # si clonaste el repositorio, usa los archivos locales

def ruta(nombre):
    """Devuelve la ruta local del archivo si existe; si no, la URL de GitHub."""
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

def pesos(x):
    """Formato de pesos colombianos: $ 1.234.567"""
    return "$ " + f"{x:,.0f}".replace(",", ".")

def millones(x):
    return "$ " + f"{x/1e6:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".") + " M"

def porc(v, signo=False):
    """Porcentaje con coma decimal: 18,9 %"""
    txt = f"{v:+.1%}" if signo else f"{v:.1%}"
    return txt.replace(".", ",").replace("%", " %")

def comprobar(respuesta, esperado, tolerancia=0.0, unidad=""):
    """Compara tu respuesta con el resultado esperado y te da pistas."""
    if respuesta is None:
        print("Aún no has escrito tu respuesta: reemplaza None por tu resultado y vuelve a ejecutar.")
        return
    if isinstance(esperado, str):
        ok = str(respuesta).strip().lower() == esperado.strip().lower()
    else:
        try:
            ok = abs(float(respuesta) - float(esperado)) <= tolerancia
        except (TypeError, ValueError):
            print("Tu respuesta debe ser un número."); return
    if ok:
        print(f"Correcto. {respuesta}{unidad} es el resultado esperado.")
    else:
        print(f"Todavía no. Revisa los filtros (¿quitaste las anuladas?) y el periodo. Tu respuesta: {respuesta}{unidad}")

# Plantilla de gráficos Dr. Bits: separador decimal coma y miles punto, como en Colombia
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=["#4A5AD8", "#0E9BB8", "#D0336F"],
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    yaxis=dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True),
    separators=",.", hoverlabel=dict(font_size=13), margin=dict(t=70, l=10, r=20, b=40),
))
pio.templates.default = "plotly_white+drbits"
def dar_aire(fig, maximo, eje="x"):
    """Deja espacio a la derecha (o arriba) para que las etiquetas de las barras no se corten."""
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

MM = "Millones de pesos"   # en español «billón» es 10^12: evitamos la abreviatura B de Plotly
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
pd.set_option("display.max_columns", 30)
pd.set_option("styler.format.decimal", ",")      # tablas con estilo: 18,9 %
pd.set_option("styler.format.thousands", ".")
print("Entorno listo. Datos desde:", "archivos locales" if os.path.exists(BASE_LOCAL) else "GitHub")

## Parte A · El reporte que llega por correo
El libro de práctica tiene la hoja **Reporte_Mal_Estructurado**: el informe de ventas 2025 tal como lo armaría alguien para imprimirlo. Léelo sin decirle a pandas dónde está el encabezado (`header=None`) para verlo tal cual: cada `NaN` es una celda vacía.

In [ ]:
reporte = pd.read_excel(ruta("Distribuidora_Andina_practica.xlsx"),
                        sheet_name="Reporte_Mal_Estructurado", header=None)
reporte.head(16)

Observa tres cosas en la salida:

* La ciudad aparece una sola vez por bloque. Las demás filas tienen `NaN` porque en Excel era una **celda combinada**: el valor solo vive en la primera celda.
* Hay filas `Subtotal …` y una fila `TOTAL GENERAL` mezcladas con los registros.
* Algunas cifras están escritas como texto, por ejemplo `'1.204,9 M'`.

### Diagnóstico con código
Antes de corregir, cuantifica. Primero encuentra dónde empieza la tabla buscando la palabra *Ciudad*: así el código funciona aunque el reporte cambie de posición.

In [ ]:
fila_enc = reporte.index[reporte.eq("Ciudad").any(axis=1)][0]
col_ciudad = reporte.columns[reporte.loc[fila_enc].eq("Ciudad")][0]
print(f"El encabezado está en la fila de Excel {fila_enc + 1}, columna {chr(65 + col_ciudad)}")

datos = reporte.iloc[fila_enc + 2:, col_ciudad:col_ciudad + 6].copy()
datos.columns = ["ciudad", "linea", "T1", "T2", "T3", "T4"]

es_subtotal = datos["linea"].astype(str).str.startswith(("Subtotal", "TOTAL"))
es_vacia = datos["linea"].isna()
valores = datos[["T1", "T2", "T3", "T4"]]
es_texto = valores.map(lambda v: isinstance(v, str))

print("Filas de subtotal o total:      ", es_subtotal.sum())
print("Filas vacías o de notas:        ", es_vacia.sum())
print("Celdas de ciudad vacías:        ", datos.loc[~es_subtotal & ~es_vacia, "ciudad"].isna().sum())
print("Cifras escritas como texto:     ", int(es_texto.values.sum()))

### ¿Cuánto se equivoca una suma sobre el reporte?
Haz lo que haría alguien con prisa: seleccionar todas las cifras y sumarlas. Excel ignora el texto al sumar, así que la «suma de Excel» deja por fuera las celdas escritas como `'x,x M'` y, además, cuenta subtotales y total general como si fueran ventas.

In [ ]:
def a_numero(v):
    """Convierte '1.204,9 M' en 1204900000. Los números se dejan igual."""
    if isinstance(v, str):
        return float(v.replace(" M", "").replace(".", "").replace(",", ".")) * 1e6
    return v

suma_excel = valores[~es_texto].apply(pd.to_numeric, errors="coerce").sum().sum()
correcta = valores[~es_subtotal & ~es_vacia].map(a_numero).astype(float).sum().sum()

print("Suma «seleccionar todo» en Excel:", millones(suma_excel))
print("Suma correcta de los registros:  ", millones(correcta))
print(f"Error: la suma rápida infla el total {suma_excel / correcta:.1f} veces".replace(".", ","))

In [ ]:
comparacion = pd.DataFrame({
    "cálculo": ["Seleccionar todo y SUMA", "Solo registros, texto convertido"],
    "valor": [suma_excel, correcta]})
fig = px.bar(comparacion, x=comparacion["valor"] / 1e6, y="cálculo", orientation="h", text=comparacion["valor"].map(millones),
             color="cálculo", color_discrete_sequence=[MAGENTA, AZUL],
             title="La misma hoja, dos totales: el reporte invita a sumar subtotales")
fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(showlegend=False, xaxis_title=MM, yaxis_title="", height=280)
dar_aire(fig, suma_excel / 1e6)
fig.show()

> **Brecha que corrige esta clave.** En auditorías de hojas reales casi todos los libros tienen errores, y muchos no están en las fórmulas sino en la estructura: rangos que no incluyen filas nuevas, subtotales que se suman otra vez, textos que parecen números.

## Parte B · Reconstruir la tabla ordenada
Una tabla ordenada (*tidy*) cumple tres reglas: cada columna es una variable, cada fila es un registro y cada tipo de registro va en su propia tabla. Vamos paso a paso. Junto a cada paso está su equivalente en Excel.

**Paso 1 · Repetir la ciudad en cada fila.** En Excel: separar celdas, *Ir a especial › Celdas en blanco*, escribir `=` + la celda de arriba y `Ctrl+Enter`.

In [ ]:
ordenada = datos.copy()
ordenada["ciudad"] = ordenada["ciudad"].ffill()
ordenada.head(8)

**Paso 2 · Quitar lo que no es registro**: subtotales, total, filas en blanco y notas. En Excel: filtrar la columna *linea* y eliminar esas filas.

In [ ]:
ordenada = ordenada[~es_subtotal & ~es_vacia]
print(len(ordenada), "filas: una por ciudad y línea")

**Paso 3 · Convertir el texto en número.** En Excel: `=VALOR(SUSTITUIR(...))` o, mejor, corregirlo en la fuente (clave 2).

In [ ]:
for t in ["T1", "T2", "T3", "T4"]:
    ordenada[t] = ordenada[t].map(a_numero).astype(float)
ordenada.head()

**Paso 4 · Llevar la variable *trimestre* a una columna.** T1…T4 son valores, no variables. En Excel esto se hace con Power Query: *Anular dinamización de columnas*. En pandas, `melt`.

In [ ]:
ordenada = ordenada.melt(id_vars=["ciudad", "linea"], var_name="trimestre", value_name="subtotal")
ordenada = ordenada.sort_values(["ciudad", "linea", "trimestre"]).reset_index(drop=True)
print(ordenada.shape)
ordenada.head(10)

Ahora la tabla tiene cuatro columnas y ninguna ambigüedad. Con ella cualquier pregunta es una línea de código, o una tabla dinámica en Excel. Por ejemplo, la venta por trimestre de cada ciudad:

In [ ]:
por_trim = ordenada.groupby(["trimestre", "ciudad"], as_index=False)["subtotal"].sum()
orden_ciudades = ordenada.groupby("ciudad")["subtotal"].sum().sort_values(ascending=False).index
fig = px.line(por_trim, x="trimestre", y=por_trim["subtotal"] / 1e6, color="ciudad", markers=True,
              category_orders={"ciudad": list(orden_ciudades)},
              color_discrete_sequence=[AZUL, TEAL, MAGENTA, "#7A5AC8", "#3E8E5E", "#B07A1E"],
              title="Venta neta 2025 por trimestre y ciudad")
fig.update_layout(yaxis_title=MM, xaxis_title="", legend_title="")
fig.show()

### Comprobar contra la fuente
Un analista no confía en un reporte: lo cuadra contra la base. Compara la tabla reconstruida con `ventas.csv` (2025, sin anuladas). Las únicas diferencias deberían ser las cifras que el reporte escribió como texto redondeado a un decimal de millón.

In [ ]:
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
productos = pd.read_csv(ruta("productos.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))

fuente = (ventas[(ventas.estado != "Anulada") & (ventas.fecha.dt.year == 2025)]
          .merge(productos[["sku", "categoria"]], on="sku")
          .merge(sedes[["id_sede", "ciudad"]], on="id_sede")
          .assign(trimestre=lambda d: "T" + d.fecha.dt.quarter.astype(str))
          .groupby(["ciudad", "categoria", "trimestre"], as_index=False)["subtotal"].sum()
          .rename(columns={"categoria": "linea", "subtotal": "subtotal_fuente"}))

cuadre = ordenada.merge(fuente, on=["ciudad", "linea", "trimestre"], how="outer", indicator=True)
cuadre["diferencia"] = cuadre["subtotal"] - cuadre["subtotal_fuente"]
print("Filas que cuadran exacto:", (cuadre["diferencia"] == 0).sum(), "de", len(cuadre))
print("Diferencia máxima:", pesos(cuadre["diferencia"].abs().max()))
cuadre[cuadre["diferencia"] != 0].head()

Las diferencias existen y son pequeñas, pero son **información perdida**: nadie puede recuperar los pesos exactos de una celda que alguien escribió como «1.204,9 M». Esa es la razón para no escribir cifras como texto.

## Parte C · Diagnóstico de calidad de una base
Antes de analizar cualquier base, revisa su forma. Estas son las preguntas que debes hacerte siempre, con su respuesta en código.

In [ ]:
print("Filas y columnas:", ventas.shape)
print("\nTipos de dato:")
print(ventas.dtypes.to_string())

In [ ]:
revision = pd.Series({
    "Celdas vacías": int(ventas.isna().sum().sum()),
    "Filas duplicadas": int(ventas.duplicated().sum()),
    "id_linea repetidos": int(ventas["id_linea"].duplicated().sum()),
    "Fecha mínima": ventas["fecha"].min().date(),
    "Fecha máxima": ventas["fecha"].max().date(),
    "Cantidades <= 0": int((ventas["cantidad"] <= 0).sum()),
    "subtotal ≠ bruto − descuento": int((ventas["subtotal"] != ventas["valor_bruto"] - ventas["valor_descuento"]).sum()),
    "total ≠ subtotal + iva": int((ventas["total"] != ventas["subtotal"] + ventas["iva"]).sum()),
})
revision.to_frame("resultado")

La base pasa las pruebas de forma. Pero la calidad también es de **significado**: la columna `estado` tiene facturas anuladas que no son venta. Ninguna prueba automática lo detecta; lo detecta quien lee el diccionario de datos.

In [ ]:
ventas["estado"].value_counts().to_frame("líneas")

In [ ]:
validas = ventas[ventas["estado"] != "Anulada"]
mensual = (validas.assign(anio=validas.fecha.dt.year.astype(str), mes=validas.fecha.dt.month)
           .groupby(["anio", "mes"], as_index=False)["subtotal"].sum())
fig = px.line(mensual, x="mes", y=mensual["subtotal"] / 1e6, color="anio", markers=True,
              color_discrete_map={"2024": GRIS, "2025": AZUL},
              title="Venta neta mensual: temporada escolar, bache de mitad de año y cierre de presupuestos")
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)), ticktext=MESES, title="")
fig.update_layout(yaxis_title=MM, legend_title="")
fig.show()

## Retos
Resuelve cada reto con lo que aprendiste. Escribe tu código en la celda gris y ejecuta *Comprobar*.

### Reto 1
¿Cuántas filas tiene la tabla `ordenada`? Explica por qué ese número tiene sentido (ciudades × líneas × trimestres).

*Pista:* `len(ordenada)`.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_1
respuesta_1 = None

In [ ]:
# @title Comprobar reto 1
comprobar(respuesta_1, ordenada['ciudad'].nunique() * ordenada['linea'].nunique() * 4, unidad=' filas')

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_1 = len(ordenada)
print(respuesta_1, '=', ordenada['ciudad'].nunique(), 'ciudades ×', ordenada['linea'].nunique(), 'líneas × 4 trimestres')

### Reto 2
Desde `ventas.csv` calcula la venta neta (subtotal) de **Medellín**, línea **Tecnología**, en el **tercer trimestre de 2025**, sin anuladas.

*Pista:* Une ventas con productos y sedes como en *Comprobar contra la fuente*, o filtra la tabla `fuente`.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_2
respuesta_2 = None

In [ ]:
# @title Comprobar reto 2
esp = fuente.query("ciudad == 'Medellín' and linea == 'Tecnología' and trimestre == 'T3'")['subtotal_fuente'].iloc[0]
comprobar(respuesta_2, esp)

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_2 = fuente.query("ciudad == 'Medellín' and linea == 'Tecnología' and trimestre == 'T3'")['subtotal_fuente'].iloc[0]
print(pesos(respuesta_2))

### Reto 3
¿Cuántas **líneas** de venta están anuladas en toda la base?

*Pista:* `value_counts()` o una comparación con `==`.

In [ ]:
# Escribe tu código aquí y guarda el resultado en respuesta_3
respuesta_3 = None

In [ ]:
# @title Comprobar reto 3
comprobar(respuesta_3, int((ventas['estado'] == 'Anulada').sum()), unidad=' líneas')

In [ ]:
# @title Ver una solución (inténtalo primero)
respuesta_3 = int((ventas['estado'] == 'Anulada').sum())
print(respuesta_3)

## Apóyate en IA
Cuando recibas un reporte con otra forma, pídele a una IA que te ayude a escribir la reconstrucción. Dale la estructura, no los datos completos:

```text
Tengo en pandas un DataFrame leído de Excel con header=None. Así se ven sus primeras 12 filas:
[pega aquí reporte.head(12).to_string()]
Escribe código pandas que: (1) encuentre la fila de encabezados buscando el texto "Ciudad",
(2) rellene hacia abajo las celdas combinadas, (3) elimine subtotales y filas vacías,
(4) convierta textos como "1.204,9 M" en números y (5) use melt para dejar una columna "trimestre".
Explica cada paso con un comentario y su equivalente en Excel.
```

> **Verifica siempre.** Ejecuta el código que te proponga sobre un caso cuyo resultado ya conoces. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Un reporte sirve para leer; una tabla ordenada sirve para analizar. No los mezcles.
* Celdas combinadas, subtotales, filas vacías y números como texto producen errores **silenciosos**.
* Cuadra siempre un resultado contra su fuente.

**Siguiente:** [Clave 5 · Buscar y cruzar sin #N/D](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-05) · [Cuaderno 05](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/05_busqueda_y_cruce.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>